# Does the memory keep a different horizon on time series than on text?

**The gap this closes.** The horizon result -- forget gate `alpha`, so memory time constant
`tau = -1/ln(1-alpha)` -- was measured on **enwik8, which is Wikipedia text**. That leaves a
claim about a *temporal* mechanism resting on *non-temporal* data. It is the paper's most
visible weakness.

This trains the identical architecture on **quantile-binned time-series corpora** and
re-measures the gate, so the horizon can be compared across data domains at matched width,
depth, context and learning rate.

### The control that makes the comparison mean something

Every corpus is mapped to 256 equal-frequency bins, which pins marginal entropy at
`log(256) = 5.545` nats for all of them. Two corpora then differ only in **temporal
dependence** -- how much the past predicts the next step -- not in how their values are
distributed. `AR_phi0.00` (white noise) and `AR_phi0.90` differ in exactly one thing.

### Two outcomes, both worth reporting

| If | Then |
|---|---|
| `alpha` **differs** across corpora | the horizon depends on what the memory reads, so a horizon measured on text says nothing about a forecasting deployment |
| `alpha` is **the same** | the horizon is a property of architecture and width, not data -- which makes the enwik8 measurement transferable and retroactively justifies the paper |

Runtime: ~40 min per corpus at 6k steps on an A100. **Run ETTm1 alone first** if you want a
direction in 40 minutes rather than 3 hours.

## 1 - Setup

In [ ]:
!pip install -q titans-pytorch
!git clone -q https://github.com/thebnbrkr/marv-titan.git /content/marv-titan 2>/dev/null || true
!git clone -q --depth 1 https://github.com/lucidrains/titans-pytorch.git /content/titans-src 2>/dev/null || true
import sys; sys.path.insert(0, '/content/marv-titan/experiments')

import torch, numpy as np, time, json, os
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE=="cuda" else "")

In [ ]:
# the harness lives in the repo so notebook and script cannot drift apart
from titans_horizon_timeseries import (
    quantize, marginal_entropy, load_ett_stream, load_ar,
    build, sample_batch, measure_alpha, val_loss, N_BINS,
)
print("helpers loaded | bins:", N_BINS, "| uniform entropy:", round(float(np.log(N_BINS)),3), "nats")

## 2 - Config

`dim=384` matches the scaled run whose horizon we are comparing against, so the only thing
that changes between this notebook and that one is **the data**.

In [ ]:
DIM      = 384        # matches the scaled enwik8 run -> only the DATA differs
STEPS    = 6000
SEQ_LEN  = 256
BATCH    = 8
LR       = 2e-4
SEG      = 8          # chunk size, so tau is in the same units as before
PASSAGE  = 1024
SEED     = 0

# reference numbers from the enwik8 runs, for comparison
REF = {"dim64_enwik8":  dict(alpha=0.741, tau_pos=5.9,  val=1.974),
       "dim384_enwik8": dict(alpha=0.227, tau_pos=31.1, val=1.248)}
print(f"memory: {DIM} -> {DIM*4} -> {DIM}  ({DIM*4} units)")
for k,v in REF.items(): print(f"  reference {k:<15} alpha {v['alpha']:.3f} | tau {v['tau_pos']:.1f} pos")

## 3 - Pre-flight: confirm the corpora differ in the right way

Marginal entropy should be **identical** (~5.545) and the corpora should differ only in
temporal structure. If entropy varies, the comparison is confounded before a single GPU
second is spent.

In [ ]:
CORPORA = {
    "ETTm1":      lambda: load_ett_stream("ETTm1"),      # real, 15-minute
    "ETTh1":      lambda: load_ett_stream("ETTh1"),      # real, hourly
    "AR_phi0.90": lambda: load_ar(0.90),                 # synthetic, predictable
    "AR_phi0.00": lambda: load_ar(0.00),                 # synthetic, white noise
}

print(f"{'corpus':<14}{'train tokens':>14}{'marg. entropy':>15}")
prepared = {}
for name, loader in CORPORA.items():
    tr, va = loader()
    prepared[name] = (tr, va)
    print(f"{name:<14}{len(tr):>14,}{marginal_entropy(tr):>15.3f}")
print(f"\nuniform maximum = {np.log(256):.3f} -> all corpora should sit at this value")

## 4 - Train and measure

One optimizer per run (rebuilding Adam mid-run resets its moment estimates and changes the
trajectory -- a bug that cost us real numbers earlier). Results checkpoint to JSON after each
corpus, so a disconnect costs one corpus rather than the session.

In [ ]:
RESULTS = "horizon_by_corpus.json"
rows = json.load(open(RESULTS)) if os.path.exists(RESULTS) else []
done = {r["corpus"] for r in rows}

for name, (tr, va) in prepared.items():
    if name in done:
        print(f"skip {name} (already done)"); continue
    torch.manual_seed(SEED); np.random.seed(SEED)
    model = build(DIM).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)     # ONE optimizer for the run
    passage = va[:PASSAGE]
    t0 = time.time()

    model.train()
    for step in range(STEPS):
        loss = model(sample_batch(tr, SEQ_LEN, BATCH).to(DEVICE), return_loss=True)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        opt.step()
        if step % 1500 == 0:
            print(f"    {name} step {step:>5}  loss {loss.item():.3f}", flush=True)

    a   = measure_alpha(model, passage, DEVICE)
    tau = -1.0/np.log(1.0-a)
    row = {"corpus": name, "alpha": a, "tau_chunks": float(tau),
           "tau_positions": float(tau*SEG), "pct_context": float(tau*SEG/SEQ_LEN*100),
           "val_loss": val_loss(model, va, SEQ_LEN, BATCH, DEVICE),
           "marginal_entropy": marginal_entropy(tr), "minutes": (time.time()-t0)/60}
    rows.append(row); json.dump(rows, open(RESULTS,"w"), indent=2)
    print(f"  -> {name}: alpha {a:.4f} | tau {row['tau_positions']:.1f} pos "
          f"({row['pct_context']:.1f}% of context) | val {row['val_loss']:.3f} "
          f"| {row['minutes']:.0f} min\n", flush=True)
    del model, opt
    if DEVICE=="cuda": torch.cuda.empty_cache()

import pandas as pd
df = pd.DataFrame(rows); df.round(4)

## 5 - Compare against the text runs

In [ ]:
import matplotlib.pyplot as plt

names = list(df.corpus) + ["enwik8 (text)"]
alphas = list(df.alpha) + [REF["dim384_enwik8"]["alpha"]]
taus   = list(df.tau_positions) + [REF["dim384_enwik8"]["tau_pos"]]
# time-series corpora in the signal hue, text in a contrasting one -- domain is the identity
cols = ["#2a78d6"]*len(df) + ["#eb6834"]

fig, ax = plt.subplots(figsize=(8.4, 4.6))
ax.set_facecolor("#fcfcfb"); fig.patch.set_facecolor("#fcfcfb")
y = np.arange(len(names))
ax.barh(y, taus, height=.6, color=cols)
ax.axvline(REF["dim64_enwik8"]["tau_pos"], color="#9a9a93", lw=1.2, ls="--")
ax.annotate("dim-64 on text (5.9)", (REF["dim64_enwik8"]["tau_pos"], len(names)-.4),
            textcoords="offset points", xytext=(6,0), fontsize=9, color="#6b6b64")
for i,(t,a) in enumerate(zip(taus, alphas)):
    ax.text(t+.6, i, f"{t:.1f} pos   alpha {a:.3f}", va="center", fontsize=9.5, color="#33332f")
ax.set_yticks(y); ax.set_yticklabels(names, fontsize=10, color="#33332f")
ax.set_xlabel("memory horizon tau (sequence positions)", fontsize=10, color="#54544c")
ax.set_title(f"Does the horizon depend on what the memory reads?  (all dim={DIM})",
             fontsize=12, color="#33332f", loc="left", pad=12)
ax.grid(True, axis="x", color="#e8e8e3", lw=.8); ax.set_axisbelow(True)
for s in ("top","right","left"): ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color("#d4d4cd"); ax.tick_params(colors="#6b6b64", labelsize=9)
ax.set_xlim(right=max(taus)*1.55)
plt.tight_layout(); plt.savefig("horizon_by_corpus.png", dpi=190); plt.show()

spread = max(alphas)/max(min(alphas),1e-9)
print(f"\nalpha across corpora: {min(alphas):.3f} - {max(alphas):.3f}  ({spread:.1f}x spread)")
print(f"for comparison, the WIDTH effect was 0.741 -> 0.227  (3.3x)")
print("\nif the corpus spread is small next to 3.3x, the horizon is set by architecture,")
print("not by data -- and the enwik8 measurement transfers to forecasting settings.")

## How to read it

**Compare the corpus spread against the width spread (3.3x).** The width effect was large and
consistent across five checkpoints. If swapping text for ETTm1 moves `alpha` by far less than
that, the horizon is a property of architecture and width -- and measuring it on text was
legitimate all along.

**Check `AR_phi0.00` against `AR_phi0.90` specifically.** Those two differ in exactly one
thing: temporal dependence. Marginal entropy is pinned identical by the binning. If `alpha`
moves between them, it is temporal structure doing it and nothing else.

**One seed.** Treat a small difference as noise. The width effect survived five checkpoints;
a corpus effect deserves the same scepticism before it goes in a paper. Re-run with
`SEED = 1, 2` if the numbers look close.

**Sanity check first.** Every corpus should show marginal entropy ~5.545 in section 3. If one
does not, its binning failed and its row means nothing.